In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_project.bronze.users (
    user_id BIGINT,
    name STRING,
    changing_value INT,
    ingested_at TIMESTAMP
) USING DELTA

In [0]:
%python
from pyspark.sql.types import LongType, StringType, StructField, StructType, IntegerType
from pyspark.sql.functions import current_timestamp

file_dir = f"/Volumes/databricks_project/generated_files/files"
table_name = f"databricks_project.bronze.users"

customSchema = StructType([
    StructField("user_id", LongType(), True),        
    StructField("name", StringType(), True),
    StructField("changing_value", IntegerType(), True)
])

df = spark.read.format("csv") \
    .option("header", "true") \
    .schema(customSchema) \
    .load(f"{file_dir}/users.csv")

df.withColumn("ingested_at", current_timestamp()) \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{table_name}")